![Week 3 Day 5](../assets/banners/w3d5.png)

# Week 3 Day 5 - 🧱 Project: Exam Understanding v2

It's project day! 🎉

Think back to week 1, day 5. GradeWise turned Emma's exam pack into an `ExamPaper` with one structured-output call. It worked, but if the marks didn't add up, all it could do was warn you, and it saved whatever the model returned without asking anyone.

This week you've learned every piece needed to do better: the **evaluator-optimiser** pattern (day 1), **state, conditional edges and loops** (day 3), **checkpointers and `interrupt()`** (day 4). Today they come together in GradeWise's **exam understanding v2**: a LangGraph workflow that extracts the exam, **validates** it, **retries with feedback** if something's wrong, and then **pauses for Emma's approval** before storing anything.

You'll run it end to end, **even without an API key**, thanks to a clearly-labelled stand-in model. Then, with a key, for real.

![Your 6-week roadmap](../assets/roadmap/week3.png)

### 🎯 By the end of this lab you will
1. 🗺️ Read GradeWise's exam graph: `gradewise/exam/graph.py`
2. ⏸️ Run it until it **pauses** for Emma, inspect the payload, and resume it with `Command(resume=...)`
3. 🔁 Watch the **self-correcting retry**: a flawed extraction goes back to the model *with its problems*
4. 💾 Pause, "restart Python", and resume with the `SqliteSaver` checkpointer
5. 🧱 Run the week's milestone script, `week3/project/exam_graph.py`

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | 🧱 Day 5 - Exam Understanding v2: Extract → Validate → Retry → Admin Approval → Store | 🧪 |
| 2 | Day 5 - Week 3 Wrap-Up: Summary, Roadmap and Quiz | 🎬 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Most of today's cells use a <b>🎭 stand-in model</b> that returns the reference answer, <code>data/exams/cs_midterm/expected_exam.json</code>, instead of calling an LLM. Everything else is <b>100% real</b>: the real PDFs are parsed, the real graph runs, the real validation, interrupt, checkpoints and database. The stand-in only replaces the one extraction call, so you can see the flow for free. Cells marked <b>🔑</b> use your real model.</span>
        </td>
    </tr>
</table>

## Imports and environment

💡 **Fast & cheap run.** Two lines in `.env` decide what this notebook costs. `MODEL_MODE=free` runs on OpenRouter's free models for $0 (same `OPENROUTER_API_KEY`, no credits, but only about 50 requests a day), and `DATA_SIZE=small` uses a small slice of the data (2 PDFs · Q1–Q3 · 2 students) when you just want to see the APIs work. Restart the kernel after editing `.env`; the next cell prints what you're running.

In [ ]:
import importlib.util
import os
from uuid import uuid4

from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from gradewise import datasets
from gradewise.config import DATA_DIR, OUTPUT_DIR, settings
from gradewise.db import list_exams, load_exam
from gradewise.exam.graph import build_exam_graph, print_exam_summary, sqlite_checkpointer, start_input
from gradewise.schemas import ExamPaper

load_dotenv(override=True)

# 🔑 cheap and free both use one key, OPENROUTER_API_KEY (from openrouter.ai/keys); custom may need others
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"ℹ️ {key_name} not set. That's OK: only the cells marked 🔑 need a key, the rest run offline.")
        print("   To set one up, see Part 5 of the setup guide (and remember to SAVE .env!)")

print(settings.describe_models())
print(datasets.describe())

EXAM_DIR = DATA_DIR / "exams" / "cs_midterm"
DEMO_DB = OUTPUT_DIR / "week3_day5_demo.db"  # 💡 the stand-in's results go to a separate demo database
print("Exam pack:", sorted(p.name for p in EXAM_DIR.iterdir()))

## Part 1: The graph, on paper

Open **`gradewise/exam/graph.py`** in your editor. It's the packaged version of this week's milestone. Here's the shape:

![Diagram](../assets/diagrams/7691c38317f6.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  S([start]) --> P[parse] --> E[extract] --> V{validate}
  V -- "problems, tries left" --> R[retry] --> E
  V -- "clean, or out of tries" --> A[[admin_review ⏸ interrupt]]
  A -- approved --> D[(store: SQLite + JSON)]
  A -- rejected --> X([end])
```

</details>


| Node | What it does | 🔁 Where you learned it |
|---|---|---|
| `parse` | Reads the three PDFs (+ `model_answers.md`) into clean text, headers and footers stripped | Week 1 day 5 |
| `extract` | One structured-output call via `extract_once(...)`. On a retry it passes `previous=` and `feedback=` | Week 1 day 3, day 1's evaluator-optimiser |
| `validate` | `ExamPaper.problems()`: do criteria add up to each question, and questions to the total? | Day 3's `check` |
| `retry` | Logs the problems, then loops back to `extract` | Day 3's loop |
| `admin_review` | `interrupt()` with a summary for Emma; applies her decision (and any `edits`) | Day 4 |
| `store` | Saves the approved exam to SQLite (`gradewise/db.py`) and `output/<exam_id>.json` | |

The public pieces you'll use: **`build_exam_graph(checkpointer=None, llm=None, *, db=None, write_json=True)`**, **`start_input(exam_id, max_attempts=3)`** and **`print_exam_summary(paper, problems)`**.

In [ ]:
# 🎭 A STAND-IN for the extraction model, so you can SEE the whole workflow offline.
# ⚠️ It does NOT read the PDFs: it hands back answers we prepared earlier (canned ExamPapers).
#    This is the same trick tests/test_exam.py uses. With a real key, the real model does this job.
from langchain_core.runnables import RunnableLambda


class FakeStructuredLLM:
    """`with_structured_output(schema)` returns the next canned answer, and records every prompt in `.calls`."""

    def __init__(self, outputs: list) -> None:
        self.outputs = list(outputs)
        self.calls: list[list] = []

    def with_structured_output(self, schema, **kwargs):
        def respond(messages):
            self.calls.append(messages)
            out = self.outputs.pop(0)
            return out if isinstance(out, schema) else schema.model_validate(out)

        return RunnableLambda(respond)


expected = ExamPaper.model_validate_json((EXAM_DIR / "expected_exam.json").read_text(encoding="utf-8"))
print(f"🎭 Stand-in ready: it will answer with expected_exam.json ({len(expected.questions)} questions)")

In [ ]:
from gradewise.display import show_graph

graph = build_exam_graph(llm=FakeStructuredLLM([expected]), db=DEMO_DB, write_json=False)
show_graph(graph)

## Part 2: Run it until it pauses ⏸️

One `invoke` runs `parse → extract → validate → admin_review`, and then stops at the `interrupt()`. Remember: the graph needs a `thread_id`, because that's where the checkpointer files the paused run.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">The stand-in returns a clean exam on the first try. Predict: how many times will <code>extract</code> run, will <code>retry</code> run at all, and what will <code>invoke</code> return: the final state, or something else?</span>
        </td>
    </tr>
</table>

In [ ]:
llm = FakeStructuredLLM([expected])
graph = build_exam_graph(llm=llm, db=DEMO_DB, write_json=False)
config = {"configurable": {"thread_id": "cs_midterm-demo"}}

result = graph.invoke(start_input("cs_midterm"), config)

print("Paused?", "__interrupt__" in result, "· waiting at:", graph.get_state(config).next)
print("Attempts:", result["attempts"], "· model calls:", len(llm.calls))

Nothing has been stored yet. The graph is **frozen** at `admin_review`, and the interrupt payload holds everything Emma needs to decide. This is what `print_exam_summary` shows her:

In [ ]:
payload = result["__interrupt__"][0].value
print(payload["message"])
print_exam_summary(ExamPaper.model_validate(payload["paper"]), payload["problems"])

In [ ]:
# 💡 Proof that the real PDFs were parsed: the prompt the (stand-in) model received
prompt = llm.calls[0][1].content
print(f"The prompt is {len(prompt):,} characters long and contains:")
for section in [
    "=== QUESTION PAPER ===",
    "=== MARKING SCHEME ===",
    "=== MARKING METHODOLOGY ===",
    "=== MODEL ANSWERS ===",
]:
    print(f"  {'✅' if section in prompt else '❌'} {section}")

### ▶️ Emma approves
She replies through `Command(resume=...)`. `admin_review` re-runs from the top, `interrupt()` returns her answer this time, and the graph carries on to `store`.

In [ ]:
final = graph.invoke(Command(resume={"approved": True}), config)

print("Stored?", final["stored"])
print("In the demo database:", [(r.exam_id, r.question_count, r.approved_by) for r in list_exams(db=DEMO_DB)])
assert load_exam("cs_midterm", db=DEMO_DB) == expected

## Part 3: The self-correcting retry 🔁

Real models make mistakes: a criterion missed here, a mark misread there. Let's make the stand-in **get it wrong first**: it "forgets" one of Q3's criteria, so Q3's criteria add up to 3 instead of 4. On the second call it returns the right answer.

In [ ]:
flawed = expected.model_copy(deep=True)
q3 = next(q for q in flawed.questions if q.id == "Q3")
q3.criteria = q3.criteria[:2]  # ⚠️ dropped Q3.c3: now Q3's criteria add up to 3, not 4
print("The flawed extraction's problems:", flawed.problems())

llm = FakeStructuredLLM([flawed, expected])
graph = build_exam_graph(llm=llm, db=DEMO_DB, write_json=False)
config = {"configurable": {"thread_id": "cs_midterm-retry"}}

for update in graph.stream(start_input("cs_midterm"), config, stream_mode="updates"):
    for node, change in update.items():
        print(f"▸ {node:13} {'⏸ paused for Emma' if node == '__interrupt__' else sorted(change or {})}")

💡 `validate` found the problem, `retry` sent it back, and the second `extract` fixed it. What did the model actually see on its second attempt? Its **previous answer**, and the **exact problem** to fix:

In [ ]:
second_attempt = llm.calls[1]
print("Message types:", [m.type for m in second_attempt])
print("\nThe last message:\n" + second_attempt[-1].content)

### ⏪ The audit trail
Every step was checkpointed, so you can see exactly what happened and when. This is the trail Emma (or an exam board) can inspect later:

In [ ]:
for snap in reversed(list(graph.get_state_history(config))):
    attempts, problems = snap.values.get("attempts", "-"), len(snap.values.get("problems") or [])
    print(f"step {snap.metadata['step']:>2} · next={str(snap.next):18} · attempts={attempts} · problems={problems}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">What if the model got it wrong <b>three times in a row</b>, with <code>max_attempts=3</code>? Does GradeWise give up and save nothing, save the flawed exam, or something else? Look at <code>after_validate</code> in <code>gradewise/exam/graph.py</code> before running the next cell.</span>
        </td>
    </tr>
</table>

In [ ]:
graph = build_exam_graph(llm=FakeStructuredLLM([flawed] * 3), db=DEMO_DB, write_json=False)
config = {"configurable": {"thread_id": "cs_midterm-stubborn"}}
result = graph.invoke(start_input("cs_midterm", max_attempts=3), config)

print("Attempts:", result["attempts"], "· still paused for Emma:", "__interrupt__" in result)
print("Problems Emma is shown:", result["__interrupt__"][0].value["problems"])

final = graph.invoke(Command(resume={"approved": False}), config)  # 🧑‍🏫 Emma rejects it
print("Stored?", final.get("stored"), "← rejected, so nothing was saved")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;"><b>A human always decides.</b> Out of tries, GradeWise doesn't quietly save a flawed exam, and it doesn't throw the work away either: it sends it to Emma <i>with the list of problems</i>. And because nothing before <code>interrupt()</code> has side effects, resuming is always safe. These are the two rules to copy into every human-in-the-loop system you build.</span>
        </td>
    </tr>
</table>

## Part 4: Pause today, approve tomorrow 💾

With the in-memory checkpointer, a paused review disappears when Python stops. With `checkpointer="sqlite"` (or `sqlite_checkpointer(path)`), it's saved in a file. Emma can close her laptop and approve in the morning, and **the model is not called again**.

In [ ]:
ckpt = OUTPUT_DIR / "week3_day5_checkpoints.db"
config = {"configurable": {"thread_id": f"cs_midterm-overnight-{uuid4().hex[:6]}"}}  # 💡 a new thread per run

evening = build_exam_graph(sqlite_checkpointer(ckpt), llm=FakeStructuredLLM([expected]), db=DEMO_DB, write_json=False)
evening.invoke(start_input("cs_midterm"), config)
print("🌙 Evening: paused at", evening.get_state(config).next)
del evening  # 💡 "close the laptop"

# ☀️ Next morning: a brand-new graph whose stand-in has NO answers left. If extract ran again, it would crash.
morning = build_exam_graph(sqlite_checkpointer(ckpt), llm=FakeStructuredLLM([]), db=DEMO_DB, write_json=False)
print("☀️ Morning: still waiting at", morning.get_state(config).next)
final = morning.invoke(Command(resume={"approved": True}), config)
print("Stored?", final["stored"], "· with zero new model calls 💰")

## Part 5: 🔑 For real

Now drop the stand-in. With no `llm=` argument, `build_exam_graph` uses `gradewise.models.chat_model(temperature=0)`, your model from `.env`. This is the real milestone: it reads the PDFs, calls your model (well under a cent with `gpt-6-luna`, $0 with `MODEL_MODE=free`), validates, and pauses for you. The extraction always reads the whole paper, even with `DATA_SIZE=small`: it's one call either way.

In [ ]:
graph = build_exam_graph(checkpointer="sqlite")  # 💡 the real model, the real database (output/gradewise.db)
config = {"configurable": {"thread_id": "cs_midterm"}}
result = graph.invoke(start_input("cs_midterm"), config)

payload = result["__interrupt__"][0].value
if payload["paper"]:
    print_exam_summary(ExamPaper.model_validate(payload["paper"]), payload["problems"])
print(f"Took {result['attempts']} attempt(s). Run the next cell to approve.")

In [ ]:
final = graph.invoke(Command(resume={"approved": True}), config)  # 🧪 TRY: {"approved": False}
print("Stored?", final.get("stored"))

## Part 6: The milestone script, `week3/project/exam_graph.py`

Open **`week3/project/exam_graph.py`**. It's the same workflow written as one readable script, the version you could have written yourself this week: a module-level `ExamState`, six small node functions, two routing functions, a `build_graph(checkpointer)`, and a `main()` with a `while "__interrupt__" in result` loop that asks Emma at the keyboard. Look for the `✋ TODO` in `admin_review`: that's this week's exercise.

Let's load it and drive it with the same stand-in. (Its model lives in a module-level `LLM` dict, set in `main()`, so we can swap it.)

In [ ]:
spec = importlib.util.spec_from_file_location("exam_graph", "project/exam_graph.py")
exam_graph = importlib.util.module_from_spec(spec)
spec.loader.exec_module(exam_graph)  # 💡 runs the definitions only: main() is behind `if __name__ == "__main__"`

exam_graph.LLM["model"] = FakeStructuredLLM([expected])  # 🎭 the stand-in again
project_graph = exam_graph.build_graph(InMemorySaver())
config = {"configurable": {"thread_id": "project-demo"}}

result = project_graph.invoke({"exam_id": "cs_midterm", "max_attempts": 3}, config)
print("⏸ Paused at:", project_graph.get_state(config).next)

# 🧑‍🏫 main() would call ask_emma() here and wait for the keyboard. We answer "no", so the demo saves nothing.
final = project_graph.invoke(Command(resume={"approved": False}), config)
print("Approved?", final["approved"])

🔑 And the real thing, from the terminal (or right here). `--auto-approve` skips the keyboard question; `--sqlite` lets you press **Ctrl+C** at the prompt and pick up the paused review later without paying for another extraction:

```bash
uv run week3/project/exam_graph.py                  # asks you to approve
uv run week3/project/exam_graph.py --auto-approve   # no questions asked
uv run week3/project/exam_graph.py --sqlite         # pause & resume across runs
uv run gradewise exam extract                       # the packaged CLI version
```

In [ ]:
!uv run project/exam_graph.py --auto-approve

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">This is the shape of almost every trustworthy AI back-office system: <b>extract → validate → self-correct → human sign-off → store</b>, with every step checkpointed. Swap "exam pack" for invoices, insurance claims, lab reports or contracts and the graph barely changes. Being able to say "it never stores anything a person hasn't approved, and it can resume a paused review without paying for the model again" is a sentence that closes deals.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">From now on, every later stage reads the approved exam from the database with <code>gradewise.db.load_exam("cs_midterm")</code>: week 4 maps handwritten answers onto its questions, and week 5 grades against its criteria. The CLI (<code>uv run gradewise exam extract</code>), the week 6 API and the dashboard all call this same <code>build_exam_graph()</code>.</span>
        </td>
    </tr>
</table>

## Week 3 wrap-up 🎓

What a week! Here's everything you can now do:

| Day | You learned | You built |
|:-:|---|---|
| 1 | Workflows vs agents, the five patterns, ReAct, tool design | A ReAct loop by hand |
| 2 | `create_agent`, memory, `response_format`, middleware | A CS201 rubric assistant with logging, limits, PII and HITL |
| 3 | State, nodes, edges, reducers, supersteps | Emma's rubric checker, with routing and a loop |
| 4 | `Send`, checkpoints, time travel, `interrupt()`, subgraphs, streaming | A parallel Q3 marker that pauses for moderation |
| 5 | 🧱 Putting it together | **Exam understanding v2** |

📝 **Quiz 3** (10 questions) is on Udemy at the end of the section. 🏋️ This week's exercise is `week3/week3_EXERCISE.ipynb`.

⏭ **Next week:** *Agentic RAG & reading handwriting*. You'll combine this week's graphs with week 2's knowledge base (retrieve, grade the evidence, rewrite the query, answer), and then teach GradeWise to read Sam's actual handwriting with a vision model.

## Recap, and where we are heading

Today you:
- 🗺️ read `gradewise/exam/graph.py` and mapped every node to a lesson from this week
- ⏸️ ran the real graph on the real PDFs until it **paused**, and resumed it with `Command(resume=...)`
- 🔁 watched a flawed extraction get sent back **with its problems**, and fixed on the second try
- 🧑‍🏫 saw that out of tries, a human still decides, and rejection stores nothing
- 💾 paused in the "evening" and approved in the "morning" with `SqliteSaver`, with no extra model calls
- 🧱 drove the milestone script, `week3/project/exam_graph.py`

⏭ **Next:** `week3/week3_EXERCISE.ipynb`, then `week4/day1.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;"><b>Today's exercise:</b> make the stand-in fail in a different way. Give it an answer whose <b>paper total</b> is wrong (set <code>total_marks</code> to 60), followed by one that still has a problem (Q5 missing its methodology), followed by the clean exam. Predict the number of attempts, stream the run with <code>stream_mode="updates"</code>, print what the model was told on <i>each</i> retry, and finally resume with <code>{"approved": True}</code> into the demo database. Solution: <code>week3/solutions/day5_SOLUTION.ipynb</code>. <br/><br/><b>The week 3 exercise</b> (<code>week3_EXERCISE.ipynb</code>): let Emma <b>edit a question during the interrupt</b>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/thankyou.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Well done!</h2>
            <span style="color:#22c55e;">Three weeks in, and GradeWise can read an exam, knows the course, and checks its own work with a human in the loop. That's a real system. Thank you for sticking with it: take a moment to enjoy that, then on to week 4!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• LangGraph human-in-the-loop: <a href="https://docs.langchain.com/oss/python/langgraph/interrupts">docs.langchain.com/oss/python/langgraph/interrupts</a><br/> • Persistence (SqliteSaver): <a href="https://docs.langchain.com/oss/python/langgraph/persistence">docs.langchain.com/oss/python/langgraph/persistence</a><br/> • <code>week3/project/README.md</code>: the milestone's own notes<br/> • <code>tests/test_exam.py</code>: see how every path through the graph is tested with a stand-in model</span>
        </td>
    </tr>
</table>